<!-- SPDX-License-Identifier: GPL-2.0-only -->
# 30 POV export

The notebook judges the export of a scene as a scene file of POV-Ray. The coordinates of the lesson are
right handed with z up and those of POV-Ray left handed with y up, and `pov_p` of `POV.bas` maps a point
(x, y, z) to <y, z, -x>. The notebook checks that mapping, and checks numerically that the scaling, the
turns and the move POV-Ray reads put every corner of a body where the views draw it.

In [ ]:
render_with_povray = True

In [ ]:
import sys, json, math
sys.path.insert(0, '..')
from harness import *

In [ ]:
build_devtool()
import re, shutil

## 1. The room of `Def_Pov`

In [ ]:
text = devtool('pov', packed())
for line in ['#include "colors.inc"', '#include "textures.inc"', 'sky_sphere { pigment { Bright_Blue_Sky } }',
             'camera { location <15, 8, -15> look_at <0, 0, 0> }', 'light_source { <5, 10, -20> color White }',
             'plane { <0, 1, 0>, 0 pigment { checker color White color Silver }']:
    check(line in text, f'the header holds {line}')
custom = devtool('pov', packed(), '1,2,3', '4,5,6')
check('camera { location <2, 3, -1> look_at <5, 6, -4> }' in custom, 'a camera of the lesson is mapped to POV-Ray')

## 2. The mapping of points, sizes and turns

A point (x, y, z) becomes <y, z, -x>, a size (a, b, c) becomes the scaling <b, c, a>, and a turn about
z by an angle becomes `rotate <0, -angle, 0>`, about y `rotate <-angle, 0, 0>` and about x
`rotate <0, 0, angle>`, written z first as the matrix of an item applies them.

In [ ]:
t = devtool('pov', packed(item('SPHERE', (1, 2, 3), (1.5, 1.5, 1.5))))
check('translate <2, 3, -1>' in t, 'the centre (1, 2, 3) is translated to <2, 3, -1>')
t = devtool('pov', packed(item('CUBOID', size=(1, 2, 3))))
check('scale <2, 3, 1>' in t, 'the edges (1, 2, 3) scale by <2, 3, 1>')
t = devtool('pov', packed(item('CUBE', rotation=(10, 20, 30))))
check('rotate <0, -30, 0> rotate <-20, 0, 0> rotate <0, 0, 10>' in t, 'the turns are written z, y, x with their mapped signs')

## 3. Every corner lands where the views draw it

POV-Ray applies `scale`, then each `rotate <a, b, c>` as a turn about x, then y, then z, each with the
usual matrix of a right handed turn of the coordinates, and `translate` last. The notebook applies the
statements of the export to the unit corners of POV-Ray and compares them with the corners of the lesson
moved by the matrix of the item and mapped by <y, z, -x>.

In [ ]:
def rot3(axis, deg):
    c, s = math.cos(math.radians(deg)), math.sin(math.radians(deg))
    return {'x': [[1, 0, 0], [0, c, -s], [0, s, c]],
            'y': [[c, 0, s], [0, 1, 0], [-s, 0, c]],
            'z': [[c, -s, 0], [s, c, 0], [0, 0, 1]]}[axis]
def apply(m, p):
    return [sum(m[i][k] * p[k] for k in range(3)) for i in range(3)]
def vec(text):
    return [float(v) for v in text.split(',')]

def pov_transform(block, p):
    for name, args in re.findall(r'(scale|rotate|translate) <([^>]*)>', block):
        v = vec(args)
        if name == 'scale':
            p = [p[0] * v[0], p[1] * v[1], p[2] * v[2]]
        elif name == 'rotate':
            for axis, angle in zip('xyz', v):
                if angle:
                    p = apply(rot3(axis, angle), p)
        else:
            p = [p[0] + v[0], p[1] + v[1], p[2] + v[2]]
    return p

def lesson_to_pov(p):
    return [p[1], p[2], -p[0]]

def world(matrix16, p):
    m = [[matrix16[c * 4 + r] for c in range(4)] for r in range(4)]
    return [sum(m[i][k] * p[k] for k in range(3)) + m[i][3] for i in range(3)]

corners = [(x, y, z) for x in (0, 1) for y in (0, 1) for z in (0, 1)]
for rotation in [(0, 0, 0), (30, 0, 0), (0, 40, 0), (0, 0, 50), (15, -35, 70), (90, 90, 90)]:
    one = packed(item('CUBOID', (1, -2, 3), (2, 3, 4), rotation))
    block = devtool('pov', one).split('// Objekt 1')[1]
    m = devtool('matrix', one, 0)
    worst = 0.0
    for c in corners:
        # The unit box of POV-Ray is the image of the unit cube of the lesson.
        got = pov_transform(block.split('box {')[1], lesson_to_pov(c))
        want = lesson_to_pov(world(m, c))
        worst = max(worst, max(abs(a - b) for a, b in zip(got, want)))
    check(worst < 1e-3, f'with the turns {rotation} every corner lands where the views draw it', worst)

## 4. The groups of the CSG and the textures

A group becomes a nested `union`, `difference`, `intersection` or `merge`. A textured group gives its
texture to its children, which then write none of their own, and an empty group is left out, since
POV-Ray takes no empty CSG.

In [ ]:
werk = devtool('example', 'werkstueck')
t = devtool('pov', ','.join(map(str, werk['packed'])))
check(re.search(r'difference \{\s*// Objekt 2: Quader\s*box', t) is not None, 'the work piece is a difference that opens with its blank, a cuboid')
check(t.count('texture { Polished_Chrome }') == 1 and 'pigment' not in t.split('difference')[1], 'the texture of the group stands once and the children carry none')
t = devtool('pov', packed(item('INTERSECTION'), item('SPHERE', parent=0), item('MERGE')))
check('intersection {' in t and 'merge {' not in t and 'Leere Gruppe' in t, 'an empty group is a comment')
t = devtool('pov', packed(item('SPHERE', texture=2)))
check('texture { Gold_Metal }' in t, 'a texture is named as POV-Ray knows it')

## 5. New bodies

A declaration becomes `#declare` ahead of every item, the inner one first, since POV-Ray reads a name
before its use, and is placed once where it stands with `object { }`, as `Neu Ende` does. An instance
is one more `object { }` with its own transformation. A name becomes an identifier of POV-Ray.

In [ ]:
scene = packed(
    item('INSTANCE', position=(5, 0, 0), source=1),
    item('DECLARE'),
    item('CUBE', parent=1),
    item('DECLARE', parent=1),
    item('SPHERE', parent=3),
)
t = devtool('pov', scene)
first, second = t.index('#declare Neu_Neu_2 ='), t.index('#declare Neu_Neu =')
check(first < second < t.index('// Objekt 1:'), 'the inner declaration comes first and both precede every item')
check(len(re.findall(r'^object [{] Neu_Neu$', t, re.M)) == 2 and 'object { Neu_Neu_2' in t, 'the declaration is placed where it stands and by the instance')
check('translate <0, 0, -5>' in t.split('// Objekt 1:')[1].split('// Objekt 2')[0], 'the instance carries its own move')

## 6. The other bodies

In [ ]:
t = devtool('pov', packed(item('DISC', size=(2, 2, 1), inner=0.5)))
check('disc { <0, 0, 0>, <0, 1, 0>, 1, 0.25' in t, 'a disc writes its hole relative to its radius')
t = devtool('pov', packed(item('PYRAMID')))
check('prism { conic_sweep' in t and 'scale <1, -1, 1> translate <0.5, 1, -0.5>' in t, 'a pyramid is a conic prism turned upright')
t = devtool('pov', packed(item('CONE')))
check('cone { <0, 0, 0>, 1, <0, 1, 0>, 0' in t, 'a cone has its apex up')
t = devtool('pov', packed(item('SPHERE', (0.1 + 0.2, -0.0, 1e-7))))
check('translate <0, 0, -0.3>' in t, 'numbers are written short, with a point and without -0')

## 7. POV-Ray reads the file

Where POV-Ray is installed, it renders every example from its exported file without an error.

In [ ]:
povray = shutil.which('povray')
if render_with_povray and povray:
    import subprocess, tempfile
    for e in devtool('examples'):
        scene = devtool('pov', ','.join(map(str, devtool('example', e['key'])['packed'])))
        with tempfile.TemporaryDirectory() as d:
            path = f'{d}/scene.pov'
            open(path, 'w').write(scene)
            r = subprocess.run([povray, '-D', '+W80', '+H60', f'+I{path}', f'+O{d}/scene.png'], capture_output=True, text=True, timeout=300)
            check(r.returncode == 0, f"POV-Ray renders the example {e['key']}", r.stderr[-2000:])
else:
    print('POV-Ray is not installed, the render is skipped')